In [ ]:
# ==============================================================================
# Cell 1: Package Installations
# ==============================================================================
!pip install --upgrade --quiet langchain langchain-community langchain-groq neo4j

In [ ]:
# ==============================================================================
# Cell 2: Imports & Environment Configuration
# ==============================================================================
import os
from langchain_community.graphs import Neo4jGraph
from langchain_groq import ChatGroq
from langchain_core.documents import Document

# Set API Keys & Database Credentials
GROQ_API_KEY = "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"
NEO4J_URI = "bolt://127.0.0.1:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "password"

os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

# Initialize Neo4j Graph Connection
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD
)

# Initialize Groq LLM
llm = ChatGroq(
    temperature=0, 
    groq_api_key=GROQ_API_KEY, 
    model_name='llama-3.1-8b-instant'
)

print("Graph Database & LLM Initialized Successfully!")


In [ ]:

# ==============================================================================
# Cell 3: Load Movies Dataset into Neo4j
# ==============================================================================
movie_query = """
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') |
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') |
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') |
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""

graph.query(movie_query)
graph.refresh_schema()
print("Movies dataset loaded!")
print("Schema Summary:")
print(graph.schema)



In [ ]:
# ==============================================================================
# Cell 4: Initialize Standard GraphCypherQAChain
# ==============================================================================
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True
)


In [ ]:

# ==============================================================================
# Cell 5: Baseline Benchmark Evaluations
# ==============================================================================
print("\n--- Test Query 1 ---")
res1 = chain.invoke({"query": "tell me count of movies of every director"})
print(f"Result: {res1['result']}\n")

print("\n--- Test Query 2 (Known Baseline Failure Case) ---")
res2 = chain.invoke({"query": "Who was the director of the movie GoldenEye"})
print(f"Result: {res2['result']}\n")